<a href="https://colab.research.google.com/github/VARositsky/university-projects/blob/main/ai-development-course/04-model-optimization/cnn_pruning_and_lstm_distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q tensorflow
!pip install -q tensorflow-model-optimization

In [ ]:
%matplotlib inline

import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Dropout, Flatten, Dense, Activation, LSTM
)
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical



170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step


In [ ]:
model = Sequential([
    Conv2D(32, 3, padding='same', activation='relu', input_shape=(32,32,3)),
    BatchNormalization(),
    Conv2D(32, 3, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2),
    Dropout(0.25),

    Conv2D(64, 3, padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(64, 3, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2),
    Dropout(0.25),

    Conv2D(128, 3, padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(128, 3, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2),
    Dropout(0.25),

    Conv2D(256, 3, padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(256, 3, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2),
    Dropout(0.25),

    Flatten(),
    Dense(10)
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.fit(X_train, Y_train, epochs=15, batch_size=64)


Epoch 1/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 31s 20ms/step - accuracy: 0.3763 - loss: 2.1129
Epoch 2/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.6089 - loss: 1.1563
Epoch 3/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.7090 - loss: 0.8425
Epoch 4/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7593 - loss: 0.6967
Epoch 5/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7859 - loss: 0.6174
Epoch 6/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8124 - loss: 0.5456
Epoch 7/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8279 - loss: 0.4958
Epoch 8/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8498 - loss: 0.4316
Epoch 9/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8668 - loss: 0.3879
Epoch 10/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8823 - loss: 0.3416
Epoch 11/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8898 - loss: 0.3153
Epoch 12/15
782/782 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
baseline_acc = model.evaluate(X_test, Y_test, verbose=0)[1]
print("Baseline CNN accuracy:", baseline_acc)


Baseline CNN accuracy: 0.8321999907493591


In [ ]:
current_weights = model.get_weights()

thresholds = [0.11, 0.15, 0.18, 0.21, 0.25]

model_pruned = tf.keras.models.clone_model(model)
model_pruned.set_weights(current_weights)

model_pruned.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

for i, threshold in enumerate(thresholds):
    print(f"\n===== Pruning iteration {i+1}/5 | threshold = {threshold} =====")

    list_of_updated_weights = []

    for layer in current_weights:
        shape = layer.shape
        flat_layer = layer.flatten()

        updated_layer = np.array(
            [0 if abs(weight) < threshold else weight for weight in flat_layer]
        )

        updated_layer = updated_layer.reshape(shape)
        list_of_updated_weights.append(updated_layer)

    model_pruned.set_weights(list_of_updated_weights)

    epochs = 5 if i == len(thresholds) - 1 else 1

    model_pruned.fit(
        X_train, Y_train,
        batch_size=64,
        epochs=epochs,
        validation_split=0.1,
        verbose=1
    )

    current_weights = model_pruned.get_weights()

    acc = model_pruned.evaluate(X_test, Y_test, verbose=0)[1]
    sparsity = np.mean([np.mean(w == 0) for w in current_weights])

    print(f"Accuracy: {acc:.4f}")
    print(f"Sparsity: {sparsity:.2%}")

final_acc = model_pruned.evaluate(X_test, Y_test, verbose=0)[1]
print("\nFinal pruned CNN accuracy:", final_acc)






===== Pruning iteration 1/5 | threshold = 0.11 =====
704/704 ━━━━━━━━━━━━━━━━━━━━ 30s 25ms/step - accuracy: 0.8441 - loss: 0.4449 - val_accuracy: 0.9198 - val_loss: 0.2306
Accuracy: 0.8368
Sparsity: 2.36%

===== Pruning iteration 2/5 | threshold = 0.15 =====
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.7710 - loss: 0.6555 - val_accuracy: 0.8614 - val_loss: 0.3944
Accuracy: 0.8183
Sparsity: 2.45%

===== Pruning iteration 3/5 | threshold = 0.18 =====
704/704 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.7205 - loss: 0.7967 - val_accuracy: 0.8410 - val_loss: 0.4691
Accuracy: 0.8109
Sparsity: 2.86%

===== Pruning iteration 4/5 | threshold = 0.21 =====
704/704 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.6577 - loss: 0.9575 - val_accuracy: 0.7890 - val_loss: 0.6213
Accuracy: 0.7656
Sparsity: 3.37%

===== Pruning iteration 5/5 | threshold = 0.25 =====
Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.5558 - loss: 1.2228 - val_accuracy: 0.7196 - val_loss: 0.83

In [ ]:
X_train_seq = X_train.reshape(-1, 32, 96)
X_test_seq  = X_test.reshape(-1, 32, 96)


class Distiller(tf.keras.Model):
    def __init__(self, student, teacher):
        super().__init__()
        self.student = student
        self.teacher = teacher

    def compile(
        self,
        optimizer,
        student_loss,
        distill_loss,
        alpha=0.1,
        temperature=5
    ):
        super().compile(optimizer=optimizer)
        self.student_loss = student_loss
        self.distill_loss = distill_loss
        self.alpha = alpha
        self.temperature = temperature

    def train_step(self, data):
        (x_img, x_seq), y = data

        teacher_logits = self.teacher(x_img, training=False)

        with tf.GradientTape() as tape:
            student_logits = self.student(x_seq, training=True)

            loss_student = self.student_loss(y, student_logits)

            loss_distill = self.distill_loss(
                tf.nn.softmax(teacher_logits / self.temperature),
                tf.nn.softmax(student_logits / self.temperature)
            ) * (self.temperature ** 2)

            loss = self.alpha * loss_student + (1 - self.alpha) * loss_distill

        grads = tape.gradient(loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))

        return {"loss": loss}


In [ ]:
def build_student(lstm_units, dense_units):
    return Sequential([
        LSTM(lstm_units, input_shape=(32,96)),
        Dense(dense_units, activation='relu'),
        Dense(10)
    ])

In [ ]:
results = []

for lstm_units in [8, 16, 32, 64, 128, 256, 512]:
    for dense_units in [16, 32, 64, 128]:
        print(f"\nStudent: LSTM={lstm_units}, Dense={dense_units}")

        student = build_student(lstm_units, dense_units)
        distiller = Distiller(student, model_pruned)

        distiller.compile(
            optimizer='adam',
            student_loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
            distill_loss=tf.keras.losses.KLDivergence(),
            alpha=0.5,
            temperature=5
        )

        distiller.fit(
            x=(X_train, X_train_seq),
            y=Y_train,
            epochs=3,
            batch_size=64,
            verbose=0
        )

        student.compile(
            optimizer='adam',
            loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
            metrics=['accuracy']
        )

        acc = student.evaluate(X_test_seq, Y_test, verbose=0)[1]
        results.append((lstm_units, dense_units, acc))

        print(f"Accuracy: {acc:.4f}")



Student: LSTM=8, Dense=16


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Accuracy: 0.3314

Student: LSTM=8, Dense=32
Accuracy: 0.3260

Student: LSTM=8, Dense=64
Accuracy: 0.3127

Student: LSTM=8, Dense=128
Accuracy: 0.3179

Student: LSTM=16, Dense=16
Accuracy: 0.3591

Student: LSTM=16, Dense=32
Accuracy: 0.3749

Student: LSTM=16, Dense=64
Accuracy: 0.3605

Student: LSTM=16, Dense=128
Accuracy: 0.3787

Student: LSTM=32, Dense=16
Accuracy: 0.3979

Student: LSTM=32, Dense=32
Accuracy: 0.4099

Student: LSTM=32, Dense=64
Accuracy: 0.4206

Student: LSTM=32, Dense=128
Accuracy: 0.4039

Student: LSTM=64, Dense=16
Accuracy: 0.4477

Student: LSTM=64, Dense=32
Accuracy: 0.4597

Student: LSTM=64, Dense=64
Accuracy: 0.4588

Student: LSTM=64, Dense=128
Accuracy: 0.4652

Student: LSTM=128, Dense=16
Accuracy: 0.4751

Student: LSTM=128, Dense=32
Accuracy: 0.4795

Student: LSTM=128, Dense=64
Accuracy: 0.4861

Student: LSTM=128, Dense=128
Accuracy: 0.5027

Student: LSTM=256, Dense=16
Accuracy: 0.5013

Student: LSTM=256, Dense=32
Accuracy: 0.5053

Student: LSTM=256, Dense=64
A

In [ ]:
print("\nLSTM units | Dense units | Accuracy")
for r in results:
    print(r)


LSTM units | Dense units | Accuracy
(8, 16, 0.3314000070095062)
(8, 32, 0.32600000500679016)
(8, 64, 0.3127000033855438)
(8, 128, 0.31790000200271606)
(16, 16, 0.35910001397132874)
(16, 32, 0.3749000132083893)
(16, 64, 0.3605000078678131)
(16, 128, 0.37869998812675476)
(32, 16, 0.3978999853134155)
(32, 32, 0.4099000096321106)
(32, 64, 0.4205999970436096)
(32, 128, 0.40389999747276306)
(64, 16, 0.44769999384880066)
(64, 32, 0.45969998836517334)
(64, 64, 0.45879998803138733)
(64, 128, 0.4652000069618225)
(128, 16, 0.47510001063346863)
(128, 32, 0.4794999957084656)
(128, 64, 0.4860999882221222)
(128, 128, 0.5026999711990356)
(256, 16, 0.5012999773025513)
(256, 32, 0.505299985408783)
(256, 64, 0.5189999938011169)
(256, 128, 0.5200999975204468)
(512, 16, 0.10000000149011612)
(512, 32, 0.5109000205993652)
(512, 64, 0.5425000190734863)
(512, 128, 0.5282999873161316)
